
<h1 id="Tutorial:-Direct-vs-Programmatic-Tool-Calling-%E2%80%94-Refund-Selection-and-Approval">Tutorial: Direct vs Programmatic Tool Calling — Refund Selection and Approval<a class="anchor-link" href="#Tutorial:-Direct-vs-Programmatic-Tool-Calling-%E2%80%94-Refund-Selection-and-Approval">¶</a></h1><p>Compare read-heavy refund candidate selection separately from approval-sensitive actions. The notebook uses deterministic local fixtures, simulated approval and refund tools, explicit live opt-ins, and quality gates before any cost comparison.</p>



<h2 id="Audience,-prerequisites,-and-learning-goals">Audience, prerequisites, and learning goals<a class="anchor-link" href="#Audience,-prerequisites,-and-learning-goals">¶</a></h2><p>This tutorial is for developers designing tool workflows that mix bulk reads with approval or write boundaries.</p>
<p>Prerequisites:</p>
<ul>
<li>Python 3.11 or newer and <code>uv</code></li>
<li>An OpenAI API key only for optional live model runs</li>
<li>Familiarity with Responses API continuation items</li>
</ul>
<p>You will learn to compare Direct and Programmatic selection fairly, keep write tools unavailable to generated code, stop before actions when selection fails, and compare All-Direct with a Hybrid workflow using quality-adjusted cost.</p>



<h2 id="Outline">Outline<a class="anchor-link" href="#Outline">¶</a></h2><ol>
<li>Configure offline-first execution.</li>
<li>Inspect delayed-order fixtures, policies, and oracles.</li>
<li>Compare read-only tool surfaces and Direct-only action tools.</li>
<li>Optionally compare candidate selection.</li>
<li>Optionally compare All-Direct and Hybrid approval workflows.</li>
<li>Inspect stage-level tokens, cost, routes, and safety gates.</li>
</ol>



<h2 id="1.-Setup">1. Setup<a class="anchor-link" href="#1.-Setup">¶</a></h2><p>Imports locate the reusable package without loading or printing credentials.</p>


In [1]:
from __future__ import annotations

import json
import os
import sys
import uuid
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ptc_benchmark.config import configured_model, load_local_environment, require_api_key
from ptc_benchmark.pricing import estimate_run_cost, load_pricing_catalog
from ptc_benchmark.refund import REFUND_SCALE_COUNTS, build_refund_approval, build_refund_selection
from ptc_benchmark.refund_evaluation import evaluate_refund_selection
from ptc_benchmark.refund_workflow import RefundWorkflowRunner, evaluate_refund_workflow
from ptc_benchmark.reporting import append_jsonl, comparison_rows, markdown_table, request_timeline
from ptc_benchmark.runner import RunConfig, ToolCallingRunner



<h3 id="Safe-execution-controls">Safe execution controls<a class="anchor-link" href="#Safe-execution-controls">¶</a></h3><p>The first flag enables only read-only candidate comparison. Approval workflow and all-scale experiments require separate opt-ins. All approval decisions and issued refunds remain deterministic local simulations even during live model runs.</p>


In [2]:
RUN_LIVE = True # Set to True to enable live API calls and associated cost.
RUN_APPROVAL_WORKFLOW = True # Set to True for simulated approval/refund actions; requires RUN_LIVE = True.
RUN_ALL_SCALES = True # Set to True for the six-run read-only scale sweep; requires RUN_LIVE = True.

SCALE = "small"  # small=4, medium=6, large=8 delayed orders
MODEL = configured_model("gpt-5.6")
REASONING_EFFORT = "low"

PRICING_PATH = Path(os.getenv("OPENAI_PRICING_PATH", PROJECT_ROOT / "pricing" / "openai_pricing_2026-08-06.json"))
RESULTS_PATH = PROJECT_ROOT / "results" / "refund_runs.jsonl"

print({
    "RUN_LIVE": RUN_LIVE,
    "RUN_APPROVAL_WORKFLOW": RUN_APPROVAL_WORKFLOW,
    "RUN_ALL_SCALES": RUN_ALL_SCALES,
    "scale": SCALE,
    "model": MODEL,
})


{'RUN_LIVE': True, 'RUN_APPROVAL_WORKFLOW': True, 'RUN_ALL_SCALES': True, 'scale': 'small', 'model': 'gpt-5.6'}



<h2 id="2.-Two-experiments,-one-safety-boundary">2. Two experiments, one safety boundary<a class="anchor-link" href="#2.-Two-experiments,-one-safety-boundary">¶</a></h2><p>The workflow is intentionally split:</p>
<ul>
<li><strong>Selection:</strong> list delayed orders, retrieve order, delivery, history, and policy records, then calculate candidates. Compare Direct with Programmatic.</li>
<li><strong>Action:</strong> request approval for every valid candidate and issue only approved refunds. Compare All-Direct with Hybrid, where Hybrid uses Programmatic only for selection and Direct for every action.</li>
</ul>
<p>The action tools are never exposed to Programmatic Tool Calling. The workflow refuses to begin approval if the selection result fails its oracle and grounding gates.</p>



<h2 id="3.-Deterministic-fixtures-and-selection-oracle">3. Deterministic fixtures and selection oracle<a class="anchor-link" href="#3.-Deterministic-fixtures-and-selection-oracle">¶</a></h2><p>Fixtures include a short delay, an already-refunded order, an ineligible digital order, and an undelivered order. Eligible candidates exercise refund caps as well as full-value refunds.</p>


In [3]:
scale_rows = []
for scale, order_count in REFUND_SCALE_COUNTS.items():
    case = build_refund_selection(scale)
    expected = case.expected_selection()
    scale_rows.append({
        "scale": scale,
        "orders": order_count,
        "candidates": len(expected["candidates"]),
        "candidate_amount_cents": expected["total_refund_amount_cents"],
    })
display(Markdown(markdown_table(scale_rows)))


scale,orders,candidates,candidate_amount_cents
small,4,2,7200
medium,6,3,9000
large,8,4,12000


In [4]:
selection = build_refund_selection(SCALE)
approval = build_refund_approval(selection)
display(Markdown("### Selection oracle\n```json\n" + json.dumps(selection.expected_selection(), indent=2) + "\n```"))
sample_order = selection.order_ids[0]
sample = {
    "order": selection.orders[sample_order],
    "delivery": selection.deliveries[sample_order],
    "history": selection.histories[selection.orders[sample_order]["customer_id"]],
    "policy": selection.policies[sample_order],
}
display(Markdown("### One complete evidence bundle\n```json\n" + json.dumps(sample, indent=2) + "\n```"))


Selection oracle ¶ { 
 "candidates" : [ 
 { 
 "order_id" : "ord-001" , 
 "customer_id" : "cus-001" , 
 "delay_hours" : 36 , 
 "refund_amount_cents" : 3000 , 
 "reason" : "delivery_delay_36_hours" , 
 "evidence_ids" : [ 
 "delivery-ord-001" , 
 "history-cus-001" , 
 "order-ord-001" , 
 "policy-ord-001" 
 ] 
 }, 
 { 
 "order_id" : "ord-003" , 
 "customer_id" : "cus-003" , 
 "delay_hours" : 10 , 
 "refund_amount_cents" : 4200 , 
 "reason" : "delivery_delay_10_hours" , 
 "evidence_ids" : [ 
 "delivery-ord-003" , 
 "history-cus-003" , 
 "order-ord-003" , 
 "policy-ord-003" 
 ] 
 } 
 ], 
 "total_refund_amount_cents" : 7200 
 }

One complete evidence bundle ¶ { 
 "order" : { 
 "evidence_id" : "order-ord-001" , 
 "order_id" : "ord-001" , 
 "customer_id" : "cus-001" , 
 "category" : "standard" , 
 "order_total_cents" : 8500 
 }, 
 "delivery" : { 
 "evidence_id" : "delivery-ord-001" , 
 "order_id" : "ord-001" , 
 "status" : "delivered" , 
 "promised_at" : "2026-08-01T12:00:00Z" , 
 "delivered_at" : "2026-08-03T00:00:00Z" , 
 "delay_hours" : 36 
 }, 
 "history" : { 
 "evidence_id" : "history-cus-001" , 
 "customer_id" : "cus-001" , 
 "refunded_order_ids" : [] 
 }, 
 "policy" : { 
 "evidence_id" : "policy-ord-001" , 
 "order_id" : "ord-001" , 
 "delay_refund_eligible" : true , 
 "minimum_delay_hours" : 24 , 
 "maximum_refund_cents" : 3000 
 } 
 }


<h3 id="Approval-oracle">Approval oracle<a class="anchor-link" href="#Approval-oracle">¶</a></h3><p>Candidate status is not approval. The fixture deliberately rejects one eligible candidate, demonstrating why selection and action must remain separate.</p>


In [5]:
display(Markdown("```json\n" + json.dumps(approval.expected_result(), indent=2) + "\n```"))


{ 
 "approvals" : [ 
 { 
 "order_id" : "ord-001" , 
 "decision" : "approved" , 
 "approval_id" : "approval-ord-001" , 
 "rationale" : "Verified carrier delay qualifies for automatic approval." 
 }, 
 { 
 "order_id" : "ord-003" , 
 "decision" : "rejected" , 
 "approval_id" : null , 
 "rationale" : "Cold-chain damage investigation is still open." 
 } 
 ], 
 "refunds" : [ 
 { 
 "order_id" : "ord-001" , 
 "approval_id" : "approval-ord-001" , 
 "refund_amount_cents" : 3000 , 
 "refund_transaction_id" : "refund-ord-001" , 
 "status" : "issued" 
 } 
 ], 
 "total_issued_cents" : 3000 
 }


<h2 id="4.-Tool-surfaces-and-caller-restrictions">4. Tool surfaces and caller restrictions<a class="anchor-link" href="#4.-Tool-surfaces-and-caller-restrictions">¶</a></h2><p>The five read tools change only <code>allowed_callers</code> between arms. The two action tools always declare <code>allowed_callers=["direct"]</code>; the approval scenario rejects any attempt to construct a Programmatic surface.</p>


In [6]:
tool_rows = []
for arm in ("direct", "programmatic"):
    for tool in selection.tool_definitions(arm):
        tool_rows.append({
            "stage": "selection",
            "arm": arm,
            "name": tool.get("name", "hosted runtime"),
            "type": tool["type"],
            "allowed_callers": tool.get("allowed_callers", []),
        })
for tool in approval.tool_definitions("direct"):
    tool_rows.append({
        "stage": "approval",
        "arm": "direct only",
        "name": tool["name"],
        "type": tool["type"],
        "allowed_callers": tool["allowed_callers"],
    })
display(Markdown(markdown_table(tool_rows)))


stage,arm,name,type,allowed_callers
selection,direct,list_delayed_orders,function,['direct']
selection,direct,get_order,function,['direct']
selection,direct,get_delivery_events,function,['direct']
selection,direct,get_refund_history,function,['direct']
selection,direct,get_refund_policy,function,['direct']
selection,programmatic,list_delayed_orders,function,['programmatic']
selection,programmatic,get_order,function,['programmatic']
selection,programmatic,get_delivery_events,function,['programmatic']
selection,programmatic,get_refund_history,function,['programmatic']
selection,programmatic,get_refund_policy,function,['programmatic']



<h2 id="5.-Orchestration-contracts">5. Orchestration contracts<a class="anchor-link" href="#5.-Orchestration-contracts">¶</a></h2><p>Selection uses identical policy math and evidence requirements. Direct reads details in parallel after listing orders; Programmatic performs the same fan-out and reduction inside JavaScript. Approval always uses Direct calls and requires a valid approval ID before each simulated refund.</p>


In [7]:
for arm in ("direct", "programmatic"):
    instructions, _ = selection.prompt(arm)
    orchestration = instructions.split("<tool_orchestration>", 1)[1].split("</tool_orchestration>", 1)[0].strip()
    display(Markdown(f"### Selection: {arm.title()}\n```text\n{orchestration}\n```"))
approval_instructions, _ = approval.prompt("direct")
approval_orchestration = approval_instructions.split("<tool_orchestration>", 1)[1].split("</tool_orchestration>", 1)[0].strip()
display(Markdown("### Approval: Direct only\n```text\n" + approval_orchestration + "\n```"))


Selection: Direct ¶ Use Direct Tool Calling. Inspect the initial order list, then issue independent detail
calls in parallel. Calculate the candidate set from returned data. Do not generate a program.

Selection: Programmatic ¶ Use Programmatic Tool Calling for the complete read and calculation stage. Retrieve the
order list, create all independent detail-call promises, resolve them with Promise.all,
and filter, calculate, and sort inside JavaScript. Emit only the result with
text(JSON.stringify(result)). Do not call refund functions directly.

Approval: Direct only ¶ Use Direct Tool Calling for all approval and refund actions. Approval and write tools
must never be invoked from generated code. Parallel approval requests are allowed;
refund calls may begin only after their corresponding approvals are observed.


<h2 id="6.-Offline-safety-and-consistency-gates">6. Offline safety and consistency gates<a class="anchor-link" href="#6.-Offline-safety-and-consistency-gates">¶</a></h2><p>Verify deterministic totals, caller restrictions, and that rejected candidates never appear in issued refunds before spending model tokens.</p>


In [8]:
selection_result = selection.expected_selection()
approval_result = approval.expected_result()
rejected = {row["order_id"] for row in approval_result["approvals"] if row["decision"] == "rejected"}
issued = {row["order_id"] for row in approval_result["refunds"]}
offline_checks = {
    "candidate_total_matches": selection_result["total_refund_amount_cents"] == sum(row["refund_amount_cents"] for row in selection_result["candidates"]),
    "issued_total_matches": approval_result["total_issued_cents"] == sum(row["refund_amount_cents"] for row in approval_result["refunds"]),
    "rejected_never_issued": rejected.isdisjoint(issued),
    "actions_direct_only": all(tool["allowed_callers"] == ["direct"] for tool in approval.tool_definitions("direct")),
}
assert all(offline_checks.values()), offline_checks
display(Markdown(markdown_table([offline_checks])))


candidate_total_matches,issued_total_matches,rejected_never_issued,actions_direct_only
True,True,True,True



<h2 id="7.-Optional-read-only-selection-comparison">7. Optional read-only selection comparison<a class="anchor-link" href="#7.-Optional-read-only-selection-comparison">¶</a></h2><p>Set <code>RUN_LIVE = True</code> to run one Direct and one Programmatic selection. This stage cannot request approval or issue a refund. Results are scored before estimated cost is compared.</p>


In [9]:
selection_results = {}

if not RUN_LIVE:
    print("Read-only live comparison skipped. Set RUN_LIVE = True to opt in.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = ToolCallingRunner(OpenAI())
    config = RunConfig(model=MODEL, reasoning_effort=REASONING_EFFORT)
    pair_id = f"refund-selection-{SCALE}-{uuid.uuid4().hex[:8]}"

    for arm in ("direct", "programmatic"):
        run = runner.run(arm=arm, scenario=selection, config=config, run_id=pair_id)
        evaluation = evaluate_refund_selection(run, selection)
        cost = estimate_run_cost(run, pricing)
        selection_results[arm] = (run, evaluation, cost)
        append_jsonl(RESULTS_PATH, run, evaluation, cost)

    display(Markdown(markdown_table(comparison_rows(selection_results.values()))))


arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
direct,True,3,17,3083,0,1875,733,92,0.039749,13.842
programmatic,True,8,17,3410,1266,2007,997,113,0.043772,24.332



<h2 id="8.-Inspect-selection-traces">8. Inspect selection traces<a class="anchor-link" href="#8.-Inspect-selection-traces">¶</a></h2><p>Programmatic should return a reduced candidate object from <code>program_output</code>; Direct receives detailed function outputs in the model loop. Both must call the same read tools exactly once per order.</p>


In [10]:
if not selection_results:
    print("No live selection traces to display.")
else:
    for arm, (run, evaluation, cost) in selection_results.items():
        display(Markdown(f"### {arm.title()} timeline"))
        display(Markdown(markdown_table(request_timeline(run))))
        display(Markdown(f"**Quality:** `{evaluation.passed}`  \n**Estimated cost:** `${cost.total_cost:.6f}`  \n**Latency:** `{run.total_latency_seconds:.3f}s`"))
    ptc_run = selection_results["programmatic"][0]
    if ptc_run.generated_programs:
        display(Markdown("### Generated selection JavaScript\n```javascript\n" + ptc_run.generated_programs[-1] + "\n```"))
    if ptc_run.program_outputs:
        display(Markdown("### Reduced program output\n```json\n" + json.dumps(ptc_run.program_outputs[-1], indent=2) + "\n```"))


Direct timeline ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, function_call",520,0,0,51,3.922
2,"reasoning, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call",685,0,0,358,4.509
3,"reasoning, message",1878,0,1875,324,5.407


Quality: True 
 Estimated cost: $0.039749 
 Latency: 13.842s

Programmatic timeline ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, program, function_call",1269,0,1135,718,12.785
2,"function_call, function_call, function_call, function_call",0,0,0,0,1.207
3,"function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call",0,0,0,0,1.313
4,function_call,0,0,0,0,0.765
5,function_call,0,0,0,0,0.985
6,function_call,0,0,0,0,1.474
7,function_call,0,0,0,0,0.837
8,"program_output, reasoning, message",2141,1266,872,279,4.965


Quality: True 
 Estimated cost: $0.043772 
 Latency: 24.332s

Generated selection JavaScript ¶ const listed = await tools . list_delayed_orders ({ 
 date_start : "2026-07-27" , 
 date_end : "2026-08-02" 
 }); 

 const orderDetails = await Promise . all ( 
 listed . orders . map (({ order_id }) => tools . get_order ({ order_id })) 
 ); 

 const customerByOrder = new Map ( 
 orderDetails . map ( o => [ o . order_id , o . customer_id ]) 
 ); 

 const nonPolicyPromises = listed . orders . flatMap (({ order_id }) => [ 
 tools . get_delivery_events ({ order_id }), 
 tools . get_refund_history ({ 
 order_id , 
 customer_id : customerByOrder . get ( order_id ) 
 }) 
 ]); 
 const nonPolicyResults = await Promise . all ( nonPolicyPromises ); 

 const eventsByOrder = new Map (); 
 const historyByOrder = new Map (); 
 for ( const item of nonPolicyResults ) { 
 if ( Object . prototype . hasOwnProperty . call ( item , "delay_hours" )) { 
 eventsByOrder . set ( item . order_id , item ); 
 } else { 
 const matchedOrderId = listed . orders . find ( 
 o => o . customer_id === item . customer_id && 
 customerByOrder . get ( o . order_id ) === item . customer_id && 
 ! historyByOrder . has ( o . order_id ) 
 ) ? . order_id ; 
 if ( matchedOrderId ) historyByOrder . set ( matchedOrderId , item ); 
 } 
 } 

 const policiesByOrder = new Map (); 
 for ( const { order_id } of listed . orders ) { 
 const policy = await tools . get_refund_policy ({ order_id }); 
 policiesByOrder . set ( order_id , policy ); 
 } 

 const detailsByOrder = new Map ( orderDetails . map ( o => [ o . order_id , o ])); 
 const candidates = listed . orders 
 . map (({ order_id }) => { 
 const order = detailsByOrder . get ( order_id ); 
 const delivery = eventsByOrder . get ( order_id ); 
 const history = historyByOrder . get ( order_id ); 
 const policy = policiesByOrder . get ( order_id ); 
 if ( 
 delivery . status !== "delivered" || 
 ! policy . delay_refund_eligible || 
 delivery . delay_hours < policy . minimum_delay_hours || 
 history . refunded_order_ids . includes ( order_id ) 
 ) return null ; 
 return { 
 order_id , 
 customer_id : order . customer_id , 
 delay_hours : delivery . delay_hours , 
 refund_amount_cents : Math . min ( 
 order . order_total_cents , 
 policy . maximum_refund_cents 
 ), 
 reason : `delivery_delay_ ${ delivery . delay_hours } _hours` , 
 evidence_ids : [ 
 order . evidence_id , 
 delivery . evidence_id , 
 history . evidence_id , 
 policy . evidence_id 
 ]. sort () 
 }; 
 }) 
 . filter ( Boolean ) 
 . sort (( a , b ) => a . order_id . localeCompare ( b . order_id )); 

 const result = { 
 candidates , 
 total_refund_amount_cents : candidates . reduce ( 
 ( sum , c ) => sum + c . refund_amount_cents , 
 0 
 ) 
 }; 
 text ( JSON . stringify ( result ));

Reduced program output ¶ { 
 "candidates" : [ 
 { 
 "order_id" : "ord-001" , 
 "customer_id" : "cus-001" , 
 "delay_hours" : 36 , 
 "refund_amount_cents" : 3000 , 
 "reason" : "delivery_delay_36_hours" , 
 "evidence_ids" : [ 
 "delivery-ord-001" , 
 "history-cus-001" , 
 "order-ord-001" , 
 "policy-ord-001" 
 ] 
 }, 
 { 
 "order_id" : "ord-003" , 
 "customer_id" : "cus-003" , 
 "delay_hours" : 10 , 
 "refund_amount_cents" : 4200 , 
 "reason" : "delivery_delay_10_hours" , 
 "evidence_ids" : [ 
 "delivery-ord-003" , 
 "history-cus-003" , 
 "order-ord-003" , 
 "policy-ord-003" 
 ] 
 } 
 ], 
 "total_refund_amount_cents" : 7200 
 }


<h2 id="9.-Optional-All-Direct-vs-Hybrid-workflow">9. Optional All-Direct vs Hybrid workflow<a class="anchor-link" href="#9.-Optional-All-Direct-vs-Hybrid-workflow">¶</a></h2><p>This section requires both <code>RUN_LIVE = True</code> and <code>RUN_APPROVAL_WORKFLOW = True</code>. Each arm reruns selection, then performs simulated approval and refund actions. Hybrid uses Programmatic selection followed by a separate Direct approval stage. Selection failure stops the workflow before any action call.</p>


In [11]:
workflow_results = {}

if not RUN_APPROVAL_WORKFLOW:
    print("Approval workflow skipped. This is the safe default.")
elif not RUN_LIVE:
    raise RuntimeError("RUN_APPROVAL_WORKFLOW requires RUN_LIVE = True.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = RefundWorkflowRunner(OpenAI())
    config = RunConfig(model=MODEL, reasoning_effort=REASONING_EFFORT)
    pair_id = f"refund-workflow-{SCALE}-{uuid.uuid4().hex[:8]}"

    for arm in ("all_direct", "hybrid"):
        run = runner.run(arm=arm, selection=selection, config=config, run_id=pair_id)
        evaluation, _, _ = evaluate_refund_workflow(run, selection, approval)
        cost = estimate_run_cost(run, pricing)
        workflow_results[arm] = (run, evaluation, cost)
        append_jsonl(RESULTS_PATH, run, evaluation, cost)

    display(Markdown(markdown_table(comparison_rows(workflow_results.values()))))


arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
all_direct,False,6,20,5132,0,1874,1016,67,0.058482,25.274
hybrid,False,11,20,5299,1266,1847,1171,178,0.058237,30.841



<h3 id="Inspect-stage-boundaries">Inspect stage boundaries<a class="anchor-link" href="#Inspect-stage-boundaries">¶</a></h3><p>Compare stage-level timelines separately. Every action-stage call must have no program caller and must be either <code>request_refund_approval</code> or <code>issue_refund</code>.</p>


In [12]:
if not workflow_results:
    print("No live workflows to display.")
else:
    for arm, (run, evaluation, cost) in workflow_results.items():
        display(Markdown(f"### {arm}: selection stage"))
        display(Markdown(markdown_table(request_timeline(run.selection_run))))
        display(Markdown(f"### {arm}: Direct approval stage"))
        display(Markdown(markdown_table(request_timeline(run.approval_run))))
        action_rows = [{
            "tool": call.name,
            "order_id": call.arguments["order_id"],
            "caller": call.caller,
            "result": call.output.get("decision", call.output.get("status")),
        } for call in run.approval_run.tool_calls]
        display(Markdown(markdown_table(action_rows)))
        display(Markdown(f"**Workflow quality:** `{evaluation.passed}`  \n**Safety boundary:** `{evaluation.safety_boundary_passed}`  \n**Estimated total cost:** `${cost.total_cost:.6f}`"))


all_direct: selection stage ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, function_call",520,0,0,52,3.093
2,"reasoning, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call",686,0,0,356,5.623
3,"reasoning, message",1877,0,1874,300,3.47


all_direct: Direct approval stage ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"function_call, function_call",534,0,0,90,2.138
2,function_call,713,0,0,38,5.682
3,message,802,0,0,180,5.267


tool,order_id,caller,result
request_refund_approval,ord-001,None,approved
request_refund_approval,ord-003,None,rejected
issue_refund,ord-001,None,issued


...

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, program, function_call",1269,0,1135,558,7.035
2,"function_call, function_call, function_call, function_call",0,0,0,0,0.981
3,"function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call",0,0,0,0,1.227
4,function_call,0,0,0,0,0.717
5,function_call,0,0,0,0,0.822
6,function_call,0,0,0,0,1.393
7,function_call,0,0,0,0,1.024
8,"program_output, reasoning, message",1981,1266,712,278,5.334


hybrid: Direct approval stage ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"function_call, function_call",534,0,0,90,2.123
2,function_call,713,0,0,38,5.613
3,"reasoning, message",802,0,0,207,4.571


tool,order_id,caller,result
request_refund_approval,ord-001,None,approved
request_refund_approval,ord-003,None,rejected
issue_refund,ord-001,None,issued


Workflow quality: False 
 Safety boundary: True 
 Estimated total cost: $0.058237


<h2 id="10.-Optional-selection-scale-sweep">10. Optional selection scale sweep<a class="anchor-link" href="#10.-Optional-selection-scale-sweep">¶</a></h2><p>Enable <code>RUN_ALL_SCALES</code> with <code>RUN_LIVE</code> to compare read-only selection across 4, 6, and 8 orders. This adds six API runs but performs no approval or refund actions.</p>


In [13]:
scale_results = []

if not RUN_ALL_SCALES:
    print("Scale sweep skipped.")
elif not RUN_LIVE:
    raise RuntimeError("RUN_ALL_SCALES requires RUN_LIVE = True.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = ToolCallingRunner(OpenAI())
    config = RunConfig(model=MODEL, reasoning_effort=REASONING_EFFORT)
    for scale in REFUND_SCALE_COUNTS:
        case = build_refund_selection(scale)
        pair_id = f"refund-scale-{scale}-{uuid.uuid4().hex[:8]}"
        for arm in ("direct", "programmatic"):
            run = runner.run(arm=arm, scenario=case, config=config, run_id=pair_id)
            evaluation = evaluate_refund_selection(run, case)
            cost = estimate_run_cost(run, pricing)
            row = comparison_rows([(run, evaluation, cost)])[0]
            scale_results.append({"scale": scale, **row})
    display(Markdown(markdown_table(scale_results)))


scale,arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
small,direct,True,3,17,3084,0,1875,733,92,0.039754,13.363
small,programmatic,True,8,17,3256,1266,1853,849,135,0.038369,17.812
medium,direct,True,3,25,3735,0,2486,997,89,0.051693,15.368
medium,programmatic,True,9,25,3321,1266,1918,948,161,0.041745,25.93
large,direct,True,3,33,4339,0,3069,1243,64,0.062821,22.62
large,programmatic,True,11,33,3330,1266,1927,1004,92,0.043482,25.338



<h2 id="11.-Interpretation">11. Interpretation<a class="anchor-link" href="#11.-Interpretation">¶</a></h2><p>Use this order:</p>
<ol>
<li>Exclude any selection or workflow that fails a quality or grounding gate.</li>
<li>Verify that no Programmatic stage can see or call an approval/write tool.</li>
<li>Compare selection tokens, requests, tool calls, and latency.</li>
<li>For workflows, compare the sum of both stages rather than only the cheap selection stage.</li>
<li>Apply the same dated price snapshot and label all dollar values as estimates.</li>
</ol>
<p>Hybrid is useful when bulk deterministic reads benefit from in-program reduction but approval and writes require an explicit model-visible boundary. It is not automatically cheaper: the extra stage and generated code have costs that must be measured.</p>



<h2 id="Exercise">Exercise<a class="anchor-link" href="#Exercise">¶</a></h2><p>Predict what happens if <code>ord-003</code> is policy-eligible but the approval tool rejects it. Identify which object may contain it, which object must not contain it, and which tool must never be called for it. Then inspect the local approval oracle above.</p>


In [14]:
candidate_ids = {row["order_id"] for row in selection.expected_selection()["candidates"]}
rejected_ids = {row["order_id"] for row in approval.expected_result()["approvals"] if row["decision"] == "rejected"}
issued_ids = {row["order_id"] for row in approval.expected_result()["refunds"]}

print("May appear in candidates:", "ord-003" in candidate_ids)
print("Rejected by approval:", "ord-003" in rejected_ids)
print("Must not appear in issued refunds:", "ord-003" not in issued_ids)
print("Therefore issue_refund must never be called for ord-003.")


May appear in candidates: True
Rejected by approval: True
Must not appear in issued refunds: True
Therefore issue_refund must never be called for ord-003.



<h2 id="Pitfalls-and-extensions">Pitfalls and extensions<a class="anchor-link" href="#Pitfalls-and-extensions">¶</a></h2><ul>
<li>Do not place approval or write tools in the Programmatic tool surface.</li>
<li>Do not continue to actions after an ungrounded or malformed selection result.</li>
<li>Do not treat policy eligibility as approval.</li>
<li>Do not issue a refund without the exact approval ID and amount.</li>
<li>Do not compare a selection-only cost with an end-to-end workflow cost.</li>
<li>Do not report a cheaper failed or unsafe run as a winner.</li>
</ul>
<p>Extensions include manual-review queues, idempotency keys, partial action failures, compensation logic, approval latency, and quality-adjusted cost over repeated runs.</p>
